# Rules-based fund selection

This notebook is the forward-looking complement to the historical backtests. Instead of asking which fund won in the past, it asks which funds look best under explicit return assumptions and simple decision rules.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

candidate_dirs = [Path.cwd(), Path.cwd() / 'copilot_analysis']
analysis_dir = next(path for path in candidate_dirs if (path / 'fund_analysis.py').exists())
if str(analysis_dir) not in sys.path:
    sys.path.append(str(analysis_dir))

from fund_analysis import DEFAULT_PORTFOLIO, fund_summary_table, future_value, liquidation_tax, load_all_funds, score_funds

In [2]:
historical = score_funds(fund_summary_table(load_all_funds()))
historical[['fund', 'cagr', 'volatility', 'max_drawdown', 'composite_score']]

,fund,cagr,volatility,max_drawdown,composite_score
0,dnb_teknologi_s,0.241950,0.168476,-0.239585,0.923077
1,swedbank_robur_tech_a,0.225285,0.225762,-0.345262,0.634615
2,finserve,0.188915,0.200800,-0.337225,0.615385
3,avanza_global,0.114681,0.155970,-0.242056,0.615385
4,avanza_zero,0.127684,0.183501,-0.319196,0.615385
5,evli,0.301023,0.366296,-0.421352,0.557692
6,ct_lux_global_tech,0.209911,0.271785,-0.380908,0.519231
7,spiltan_aktiefond_investmentbolag,0.126661,0.197656,-0.324147,0.519231
8,bgf_tech_a2,0.154864,0.222312,-0.488931,0.423077
9,nordea_alfa,0.077227,0.174459,-0.365267,0.365385


In [6]:
expected_returns = pd.DataFrame(
    {
        'fund': [
            'avanza_global',
            'avanza_zero',
            'spiltan_aktiefond_investmentbolag',
            'swedbank_robur_tech_a',
            'dnb_teknologi_s',
            'nordea_alfa',
        ],
        'expected_annual_return': [0.07, 0.065, 0.075, 0.08, 0.08, 0.06],
        'conviction_weight': [1.0, 0.9, 0.9, 0.7, 0.7, 0.4],
    }
)
expected_returns

,fund,expected_annual_return,conviction_weight
0,avanza_global,0.070,1.0
1,avanza_zero,0.065,0.9
2,spiltan_aktiefond_investmentbolag,0.075,0.9
3,swedbank_robur_tech_a,0.080,0.7
4,dnb_teknologi_s,0.080,0.7
5,nordea_alfa,0.060,0.4


In [7]:
horizon_years = 30
capital_to_allocate = DEFAULT_PORTFOLIO['market_value'] - liquidation_tax(
    market_value=DEFAULT_PORTFOLIO['market_value'],
    cost_basis=DEFAULT_PORTFOLIO['cost_basis'],
    tax_rate=0.30,
)

rules_based = historical.merge(expected_returns, on='fund', how='inner')
rules_based['terminal_value_estimate'] = rules_based['expected_annual_return'].map(
    lambda annual_return: future_value(capital_to_allocate, annual_return, horizon_years)
)
rules_based['expected_return_score'] = rules_based['expected_annual_return'].rank(pct=True)
rules_based['conviction_score'] = rules_based['conviction_weight'].rank(pct=True)
rules_based['final_score'] = (
    0.4 * rules_based['composite_score']
    + 0.4 * rules_based['expected_return_score']
    + 0.2 * rules_based['conviction_score']
)
rules_based = rules_based.sort_values('final_score', ascending=False)
rules_based[['fund', 'expected_annual_return', 'terminal_value_estimate', 'final_score']]

,fund,expected_annual_return,terminal_value_estimate,final_score
0,dnb_teknologi_s,0.080,5.996683e+06,0.819231
1,swedbank_robur_tech_a,0.080,5.996683e+06,0.703846
2,avanza_global,0.070,4.536404e+06,0.646154
4,spiltan_aktiefond_investmentbolag,0.075,5.217378e+06,0.624359
3,avanza_zero,0.065,3.941728e+06,0.529487
5,nordea_alfa,0.060,3.422744e+06,0.246154


In [8]:
fig = px.bar(
    rules_based,
    x='fund',
    y='final_score',
    color='expected_annual_return',
    title='Rules-based ranking of candidate funds'
)
fig.show()

## How to use this notebook

Treat the editable expected return table as the place where your own beliefs enter the process. The historical statistics are inputs, not the whole decision.